# 54. 다인 장면에서 직전 박스로 스내치 수행자를 연결할 수 있을까?

원본 YOLO26n-pose, 입력 640, MPS, 사람 점수 0.25와 관절 표시 기준 0.50을 고정합니다. 기존 방식은 후보가 한 명일 때만 표시합니다. 실험 방식은 사람이 여러 명일 때 직전 프레임 선택 박스와 IoU 0.50 이상인 후보가 하나일 때만 연결합니다. 두 후보가 같은 사람 위에 겹칠 수 있어 후보 수와 실제 사람 수를 구분합니다.

In [ ]:
import pandas as pd
from IPython.display import display
from wlcv.evaluation.performer_continuity import PARENT, PRIMARY, OTHERS, REFERENCE, analyze, plot_review, score

run_again = False
primary = PARENT / "20261125-212609-365706"
others = PARENT / "20261125-212726-999316"
if run_again or not primary.exists():
    primary = analyze(PRIMARY)
if run_again or not others.exists():
    others = analyze(OTHERS)

### b02 수행자 판독과 연결 결과

참고 박스는 이 실험에서 예측을 열기 전 원본을 보고 작업 보조자가 표시한 근사 외곽입니다. 독립적인 트레이너 재판독은 수행하지 않았습니다. 그림에서 점선은 참고 박스, 초록색은 선택, 보라색은 다른 후보입니다. 두 방식에서 복구·보류를 같은 장면으로 비교합니다.

In [ ]:
def compare(folder, label_name):
    ref = REFERENCE / label_name
    display(score(folder, ref))
    display(plot_review(folder, ref))

compare(primary, "b02.csv")

In [ ]:
first = pd.read_csv(primary / "frames.csv")
active = first[first.in_motion]
display(active.groupby("video_id")[["single_angle", "linked_angle"]].count())
endpoints = first[first.video_id.eq("b02_sn_h_02") & first.frame_index.isin([20, 30])]
display(endpoints[["frame_index", "count", "single_angle", "linked_angle"]].round(2))
print("이륙→무릎 통과 모델 변화량(°)", round(endpoints.linked_angle.diff().iloc[-1], 2))

### 같은 규칙을 다른 선수에게 적용

나머지 검증 선수 네 명에서 동작 중 다인 후보가 나온 모든 프레임을 확인합니다. 참고 박스에 여러 후보가 겹치는 경우도 있습니다. 한 사람의 자세 좌표가 출력돼도 해부학적 정확도는 별도로 검증해야 합니다.

In [ ]:
compare(others, "other_validation.csv")
extra = pd.read_csv(others / "frames.csv")
display(extra[extra.in_motion].groupby("video_id")[["single_angle", "linked_angle"]].count())